# Non-rigid Alignment Baseline Comparison

**Dataset**: Mouse Embryo E9.5 (Slice 32 vs Slice 33)  
**Methods**: No-align, PASTE, SPACEL, STalign, Spateo (Rigid + Nonrigid), Ours (Rigid + Spatial)  
**Metrics**: OT Accuracy, NN Accuracy, Ratio, CLC Score

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('matplotlib').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import scanpy as sc
import spateo as st
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.spatial
import scipy.spatial.distance
from scipy.spatial import cKDTree
import ot
import time
import anndata as ad

%config InlineBackend.print_figure_kwargs={'facecolor' : 'w'}
%config InlineBackend.figure_format='retina'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Global seed set to 0
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Device: cuda
GPU: NVIDIA GeForce RTX 5080


## 1. Load Data

In [2]:
# Mouse Embryo E9.5 — already downloaded by align1.ipynb
slice1 = st.read('basic_usage_demo_1.h5ad')
slice2 = st.read('basic_usage_demo_2.h5ad')

LABEL_KEY = 'cellbin_SpatialDomain'
print(f'Slice1: {slice1.n_obs} cells, Slice2: {slice2.n_obs} cells')
print(f'Labels: {slice1.obs[LABEL_KEY].nunique()} types in slice1, {slice2.obs[LABEL_KEY].nunique()} types in slice2')

Slice1: 17425 cells, Slice2: 19939 cells
Labels: 16 types in slice1, 16 types in slice2


## 2. Evaluation Metrics

In [3]:
def mapping_accuracy_nn_bidi(labels1, labels2, coords1, coords2, k=1):
    """Bidirectional NN accuracy + collapse ratio."""
    l1 = np.asarray(labels1)
    l2 = np.asarray(labels2)
    c1 = np.asarray(coords1, dtype=np.float64)
    c2 = np.asarray(coords2, dtype=np.float64)

    # Forward: slice2 -> slice1
    tree1 = cKDTree(c1)
    _, idx_fwd = tree1.query(c2, k=k)
    acc_fwd = np.mean(l2 == l1[idx_fwd])

    # Backward: slice1 -> slice2
    tree2 = cKDTree(c2)
    _, idx_bwd = tree2.query(c1, k=k)
    acc_bwd = np.mean(l1 == l2[idx_bwd])

    accuracy = (acc_fwd + acc_bwd) / 2.0

    # Ratio (collapse measure)
    n_min = min(len(l1), len(l2))
    unique_fwd = len(set(idx_fwd.tolist()))
    unique_bwd = len(set(idx_bwd.tolist()))
    n_unique = (unique_fwd + unique_bwd) / 2.0
    ratio = abs(np.log2(n_min / (n_unique + 1e-8)))

    return accuracy, ratio


def mapping_accuracy_ot(labels1, labels2, coords1, coords2, max_samples=5000, seed=42):
    """OT (EMD) based accuracy with optional subsampling."""
    l1 = np.asarray(labels1)
    l2 = np.asarray(labels2)
    c1 = np.asarray(coords1, dtype=np.float64)
    c2 = np.asarray(coords2, dtype=np.float64)

    n1, n2 = len(l1), len(l2)
    rng = np.random.default_rng(seed)

    if n1 > max_samples or n2 > max_samples:
        idx1 = rng.choice(n1, min(n1, max_samples), replace=False)
        idx2 = rng.choice(n2, min(n2, max_samples), replace=False)
        c1, l1 = c1[idx1], l1[idx1]
        c2, l2 = c2[idx2], l2[idx2]

    m1, m2 = len(l1), len(l2)
    label_match = (l1[:, None] == l2[None, :]).astype(np.float64)
    cost = scipy.spatial.distance.cdist(c1, c2).astype(np.float64)
    a = np.ones(m1, dtype=np.float64) / m1
    b = np.ones(m2, dtype=np.float64) / m2
    pi = ot.emd(a, b, cost)
    return float(np.sum(pi * label_match))


def calculate_clc(labels1, labels2, coords1, coords2, k_percent=0.025):
    """Contextual Label Consistency score."""
    l1 = np.asarray(labels1)
    l2 = np.asarray(labels2)
    c1 = np.asarray(coords1, dtype=np.float64)
    c2 = np.asarray(coords2, dtype=np.float64)

    n1, n2 = len(c1), len(c2)
    k = max(1, int(((n1 + n2) / 2) * k_percent))

    tree1 = cKDTree(c1)
    _, idx_map = tree1.query(c2, k=1)
    label_match = (l1[idx_map] == l2)

    tree2 = cKDTree(c2)
    _, nbr_moving = tree2.query(c2, k=k+1)
    nbr_moving = nbr_moving[:, 1:]

    _, nbr_fixed = tree1.query(c1, k=k+1)
    nbr_fixed = nbr_fixed[:, 1:]

    scores = np.zeros(n2)
    for i in range(n2):
        if not label_match[i]:
            continue
        i_prime = idx_map[i]
        j_primes = idx_map[nbr_moving[i]]
        fixed_nbrs = set(nbr_fixed[i_prime])
        scores[i] = np.sum(np.isin(j_primes, list(fixed_nbrs))) / k

    return float(np.mean(scores))


def evaluate_all(slice1, slice2, coords2_aligned, label_key=LABEL_KEY, max_ot_samples=5000, seed=42):
    """Compute all 4 metrics."""
    coords1 = np.asarray(slice1.obsm['spatial'])
    coords2 = np.asarray(coords2_aligned)
    l1 = np.asarray(slice1.obs[label_key])
    l2 = np.asarray(slice2.obs[label_key])

    acc_ot = mapping_accuracy_ot(l1, l2, coords1, coords2, max_samples=max_ot_samples, seed=seed)
    acc_nn, ratio = mapping_accuracy_nn_bidi(l1, l2, coords1, coords2)
    clc = calculate_clc(l1, l2, coords1, coords2)

    return {'OT_Accuracy': acc_ot, 'NN_Accuracy': acc_nn, 'Ratio': ratio, 'CLC': clc}


print('Metrics defined.')

Metrics defined.


## 3. Preprocessing

In [4]:
# Keep raw copies for baselines that need unprocessed data
slice1_raw = slice1.copy()
slice2_raw = slice2.copy()

# Preprocess for methods that need it (Spateo, Ours)
for s in [slice1, slice2]:
    sc.pp.filter_cells(s, min_genes=10)
    sc.pp.filter_genes(s, min_cells=3)
    s.layers['counts'] = s.X.copy()
    sc.pp.normalize_total(s)
    sc.pp.log1p(s)
    sc.pp.highly_variable_genes(s, n_top_genes=2000)

st.align.group_pca([slice1, slice2], pca_key='X_pca')

coords1_orig = np.asarray(slice1.obsm['spatial'])
coords2_orig = np.asarray(slice2.obsm['spatial'])

# Normalize coordinates
all_c = np.vstack([coords1_orig, coords2_orig])
c_mean, c_std = all_c.mean(0), all_c.std(0) + 1e-8
c1_norm = ((coords1_orig - c_mean) / c_std).astype(np.float32)
c2_norm = ((coords2_orig - c_mean) / c_std).astype(np.float32)

print(f'Preprocessed: Slice1={slice1.n_obs}, Slice2={slice2.n_obs}')

Preprocessed: Slice1=17425, Slice2=19939


## 4. Run All Methods

In [5]:
results = {}  # method_name -> {'coords2_aligned': array, 'time': float, 'metrics': dict}

### 4.1 No-align (Baseline)

In [6]:
t0 = time.time()
coords2_noalign = coords2_orig.copy()
t_noalign = time.time() - t0

metrics_noalign = evaluate_all(slice1, slice2, coords2_noalign)
results['No-align'] = {'coords2': coords2_noalign, 'time': t_noalign, 'metrics': metrics_noalign}
print(f'No-align: {metrics_noalign}')

No-align: {'OT_Accuracy': 0.10360000000000082, 'NN_Accuracy': 0.17249540366410235, 'Ratio': 11.766942937870395, 'CLC': 0.014833249977742609}


### 4.2 PASTE

In [ ]:
try:
    import paste as pst

    s1_p = slice1.copy()
    s2_p = slice2.copy()
    # PASTE needs raw counts
    if 'counts' in s1_p.layers:
        s1_p.X = s1_p.layers['counts'].copy()
    if 'counts' in s2_p.layers:
        s2_p.X = s2_p.layers['counts'].copy()

    t0 = time.time()
    pi_paste = pst.pairwise_align(s1_p, s2_p, alpha=0.1, use_gpu=True, backend=ot.backend.TorchBackend(), verbose=False)
    t_paste = time.time() - t0

    # PASTE gives a transport plan, not aligned coords.
    # We use the plan to compute a weighted centroid for each source cell.
    pi_dense = np.array(pi_paste.todense()) if hasattr(pi_paste, 'todense') else np.array(pi_paste)
    # Normalize rows
    row_sums = pi_dense.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    pi_norm = pi_dense / row_sums
    coords2_paste = pi_norm @ coords1_orig  # map slice2 onto slice1's space

    metrics_paste = evaluate_all(slice1, slice2, coords2_paste)
    results['PASTE'] = {'coords2': coords2_paste, 'time': t_paste, 'metrics': metrics_paste}
    print(f'PASTE ({t_paste:.1f}s): {metrics_paste}')
except Exception as e:
    print(f'PASTE failed: {e}')

gpu is available, using gpu.


### 4.3 SPACEL

In [ ]:
import subprocess, tempfile

try:
    with tempfile.TemporaryDirectory() as tmpdir:
        s1_path = os.path.join(tmpdir, 's1.h5ad')
        s2_path = os.path.join(tmpdir, 's2.h5ad')
        out_path = os.path.join(tmpdir, 'result.npz')

        slice1.write_h5ad(s1_path)
        slice2.write_h5ad(s2_path)

        runner = os.path.join(os.getcwd(), 'spacel_runner.py')
        cmd = [
            'conda', 'run', '-n', 'spacel', '--no-capture-output',
            'python', runner,
            '--slice1', s1_path,
            '--slice2', s2_path,
            '--label_key', LABEL_KEY,
            '--output', out_path,
        ]

        t0 = time.time()
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        if proc.returncode != 0:
            raise RuntimeError(f'SPACEL failed:\n{proc.stderr}')

        data = np.load(out_path)
        coords2_spacel = data['coords2']
        t_spacel = float(data['elapsed'])

    metrics_spacel = evaluate_all(slice1, slice2, coords2_spacel)
    results['SPACEL'] = {'coords2': coords2_spacel, 'time': t_spacel, 'metrics': metrics_spacel}
    print(f'SPACEL ({t_spacel:.1f}s): {metrics_spacel}')
except Exception as e:
    print(f'SPACEL failed: {e}')

### 4.4 STalign

In [ ]:
try:
    from STalign import STalign as STalign_module

    xJ = np.array(coords1_orig[:, 0])
    yJ = np.array(coords1_orig[:, 1])
    XJ, YJ, J, _ = STalign_module.rasterize(xJ, yJ, dx=30)

    xI = np.array(coords2_orig[:, 0])
    yI = np.array(coords2_orig[:, 1])
    XI, YI, I, _ = STalign_module.rasterize(xI, yI, dx=30)

    t0 = time.time()
    params = {'niter': 10000, 'device': device, 'epV': 50}
    out = STalign_module.LDDMM([YI, XI], I, [YJ, XJ], J, **params)
    A, v, xv = out['A'], out['v'], out['xv']

    points_np = np.stack([yI, xI], axis=1)
    points_tensor = torch.tensor(points_np, dtype=A.dtype).to(device)
    tpointsI = STalign_module.transform_points_source_to_target(xv, v, A, points_tensor)
    if tpointsI.is_cuda:
        tpointsI = tpointsI.cpu()
    t_stalign = time.time() - t0

    coords2_stalign = tpointsI[:, [1, 0]].numpy()

    metrics_stalign = evaluate_all(slice1, slice2, coords2_stalign)
    results['STalign'] = {'coords2': coords2_stalign, 'time': t_stalign, 'metrics': metrics_stalign}
    print(f'STalign ({t_stalign:.1f}s): {metrics_stalign}')
except Exception as e:
    print(f'STalign failed: {e}')

### 4.5 Spateo (Rigid + Nonrigid)

In [ ]:
try:
    from spateo.alignment.methods import Morpho_pairwise

    s1_sp = slice1.copy()
    s2_sp = slice2.copy()

    t0 = time.time()
    morph = Morpho_pairwise(
        sampleA=s1_sp, sampleB=s2_sp,
        spatial_key='spatial', key_added='align_spatial',
        rep_layer='X_pca', rep_field='obs',
        use_gpu=True,
    )
    morph.run()
    t_spateo = time.time() - t0

    coords2_rigid_sp = np.asarray(s2_sp.obsm['align_spatial'])
    coords2_nonrigid_sp = np.asarray(s2_sp.obsm.get('align_spatial_nonrigid', s2_sp.obsm['align_spatial']))

    # Check for nonrigid key variants
    for key in ['align_spatial_nonrigid', 'Nonrigid_align_spatial']:
        if key in s2_sp.obsm:
            coords2_nonrigid_sp = np.asarray(s2_sp.obsm[key])
            break

    metrics_sp_rigid = evaluate_all(slice1, slice2, coords2_rigid_sp)
    results['Spateo_Rigid'] = {'coords2': coords2_rigid_sp, 'time': t_spateo, 'metrics': metrics_sp_rigid}
    print(f'Spateo Rigid ({t_spateo:.1f}s): {metrics_sp_rigid}')

    metrics_sp_nonrigid = evaluate_all(slice1, slice2, coords2_nonrigid_sp)
    results['Spateo_Nonrigid'] = {'coords2': coords2_nonrigid_sp, 'time': t_spateo, 'metrics': metrics_sp_nonrigid}
    print(f'Spateo Nonrigid ({t_spateo:.1f}s): {metrics_sp_nonrigid}')
except Exception as e:
    print(f'Spateo failed: {e}')

### 4.6 Ours (INST-Align)

In [ ]:
import sys
sys.path.insert(0, os.getcwd())

from inr_align.model import DeformationNet, adaptive_icp, UnifiedCostMatcher as OurMatcher
from inr_align.train import train as inr_train, apply_model
from inr_align.config import PipelineConfig
from inr_align.utils import detect_grid_spacing

# ---- Config: manually set all params, don't rely on defaults ----
config = PipelineConfig()
# Training
config.train.epochs = 300
config.train.batch_size = 2500
config.train.lam_jacobian = 0.01
config.train.warmup_fraction = 0.4
config.train.topk = 64
config.train.lr = 1e-3
config.train.scheduler_patience = 50
# Snap controls (align with current code)
config.train.snap_to_grid_training = False
config.train.snap_to_grid_inference = True
config.train.disable_snap_when_cpd = True

# Matcher
config.matcher.tau_init = 0.1
config.matcher.tau_min = 0.05
config.matcher.tau_max = 1.0
config.matcher.sinkhorn_iters = 0
# Optional: CPD outlier (set >0 to enable)
config.matcher.outlier_weight = 0.0

# ICP: PCA mode for non-grid data
config.icp.mode = "pca"

# Grid detection
sx, sy, is_grid, origin = detect_grid_spacing(c1_norm)

# ICP rigid alignment (PCA mode)
t0 = time.time()
R, t_vec, angle, rmse = adaptive_icp(
    c1_norm, c2_norm,
    config=config.icp,
    verbose=True,
    emb_A=slice1.obsm['X_pca'].astype(np.float32),
    emb_B=slice2.obsm['X_pca'].astype(np.float32),
)
c2_rigid = ((R @ c2_norm.T).T + t_vec).astype(np.float32)
coords2_rigid_ours = c2_rigid * c_std + c_mean

# GPU tensors
x1 = torch.tensor(c1_norm, device=device)
x2 = torch.tensor(c2_rigid, device=device)
emb1 = torch.tensor(slice1.obsm['X_pca'].astype(np.float32), device=device)
emb2 = torch.tensor(slice2.obsm['X_pca'].astype(np.float32), device=device)

# Deformation network
model = DeformationNet(
    config.model,
    grid_mode=is_grid,
    grid_spacing=[sx, sy] if is_grid else None,
    grid_origin=origin,
).to(device)

matcher = OurMatcher(config.matcher)

# Train
result = inr_train(model, matcher, x1, emb1, x2, emb2, config.train)

# Apply (mirror run.align_pair logic)
model.eval()
use_snap = bool(is_grid and config.train.snap_to_grid_inference)
if use_snap and config.matcher.outlier_weight > 0 and config.train.disable_snap_when_cpd:
    use_snap = False
x2_def = apply_model(model, x2, snap_to_grid=use_snap)
coords2_spatial_ours = x2_def.cpu().numpy() * c_std + c_mean
t_ours = time.time() - t0

# Evaluate rigid
metrics_ours_rigid = evaluate_all(slice1, slice2, coords2_rigid_ours)
results['Ours_Rigid'] = {'coords2': coords2_rigid_ours, 'time': t_ours, 'metrics': metrics_ours_rigid}
print(f'Ours Rigid: {metrics_ours_rigid}')

# Evaluate spatial
metrics_ours_spatial = evaluate_all(slice1, slice2, coords2_spatial_ours)
results['Ours_Spatial'] = {'coords2': coords2_spatial_ours, 'time': t_ours, 'metrics': metrics_ours_spatial}
print(f'Ours Spatial ({t_ours:.1f}s): {metrics_ours_spatial}')



## 5. Results Summary

In [ ]:
# Build summary table — include ALL methods that successfully ran
method_order = ['No-align', 'PASTE', 'SPACEL', 'STalign', 'Spateo_Rigid', 'Spateo_Nonrigid', 'Ours_Rigid', 'Ours_Spatial']

rows = []
for method in method_order:
    if method in results:
        r = results[method]
        row = {'Method': method, 'Time(s)': r['time']}
        row.update(r['metrics'])
        rows.append(row)

# Also include any methods not in the predefined order
for method in results:
    if method not in method_order:
        r = results[method]
        row = {'Method': method, 'Time(s)': r['time']}
        row.update(r['metrics'])
        rows.append(row)

df = pd.DataFrame(rows)
df = df.set_index('Method')

# Format for display
print('\n' + '=' * 80)
print('BASELINE COMPARISON — Mouse Embryo E9.5 (Slice32 vs Slice33)')
print('=' * 80)
print(df.to_string(float_format='%.4f'))
print(f'\nMethods evaluated: {len(df)}')
print('Note: OT_Accuracy, NN_Accuracy, CLC = higher is better. Ratio = lower is better.')

df

In [ ]:
# Bar chart comparison
metrics_to_plot = ['OT_Accuracy', 'NN_Accuracy', 'CLC']
colors = {
    'No-align': '#999999', 'PASTE': '#e6a532', 'SPACEL': '#5ba355',
    'STalign': '#d35b5b', 'Spateo_Rigid': '#7caed6', 'Spateo_Nonrigid': '#4a86b8',
    'Ours_Rigid': '#d98cd9', 'Ours_Spatial': '#9933cc',
}

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 5))

for ax, metric in zip(axes, metrics_to_plot):
    methods = df.index.tolist()
    vals = df[metric].values
    cs = [colors.get(m, '#333333') for m in methods]
    bars = ax.bar(range(len(methods)), vals, color=cs)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: baseline_comparison.png')

## 6. Alignment Visualization

In [ ]:
# Overlay visualization for each method
vis_methods = [m for m in method_order if m in results]
n_vis = len(vis_methods)

fig, axes = plt.subplots(1, n_vis, figsize=(4 * n_vis, 4))
if n_vis == 1:
    axes = [axes]

for ax, method in zip(axes, vis_methods):
    c1 = coords1_orig
    c2 = results[method]['coords2']
    ax.scatter(c1[:, 0], c1[:, 1], s=0.3, c='blue', alpha=0.3, label='Slice1')
    ax.scatter(c2[:, 0], c2[:, 1], s=0.3, c='red', alpha=0.3, label='Slice2')
    ax.set_title(method, fontsize=11, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

axes[0].legend(markerscale=10, fontsize=8, loc='upper left')
plt.suptitle('Alignment Overlay: Blue=Slice1, Red=Slice2 (aligned)', fontsize=12)
plt.tight_layout()
plt.savefig('alignment_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: alignment_overlay.png')

## 7. Export Results

In [ ]:
# Save results to CSV
df.to_csv('mouse_embryo_baseline_results.csv')
print('Saved: mouse_embryo_baseline_results.csv')
print('\nFinal Results:')
df